CURRENCY CONVERSION TOOL 
Currency conversion changes on the hourly basis every hour of every country 
If we go to llm and say to convert , it will not have the latest data to convert , soo we are solving this problem 
ex : 80 USD -> INR ?  (LATEST)

Its a py func where we hit the ExchangeRate API -> have to give source currency and target currency and will give the conversion factor 

After getting factor we can convert the curresncies


In [ ]:
import requests
from langchain_core.tools import tool
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")
exchange_rate_api_key = os.getenv("EXCHANGE_RATE_API_KEY")

In [ ]:
#TOOL CREATE 
#2 tools -> 1st api hit to get currency conversion factor and 2nd give answer by doing multiplication
#80 USD -> INR : Get conversion factor -> 93 and and then multiply 93x80 
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency : str, target_currency : str) -> float :
    """
    This function fetches the currency factor between a given base currency and a target currency
    """

    url = f'https://v6.exchangerate-api.com/v6/{exchange_rate_api_key}/pair/{base_currency}/{target_currency}'


    response = requests.get(url)

    return response.json()


@tool
def convert(base_currency_value : int, conversion_rate : Annotated[float, InjectedToolArg]) -> float :
    #Annotated[float, InjectedToolArg] means -> when llm calls this function then it will not set conversion rate, developers needs to inject this value after running earlier tools 
    """
    Given a currency conversion rate this function calculates the target currency value from a given base currency value
    """

    return base_currency_value*conversion_rate


In [66]:
get_conversion_factor.invoke({'base_currency' : 'USD' , 'target_currency' : 'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1783382402,
 'time_last_update_utc': 'Tue, 07 Jul 2026 00:00:02 +0000',
 'time_next_update_unix': 1783468802,
 'time_next_update_utc': 'Wed, 08 Jul 2026 00:00:02 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.4559}

In [67]:
convert.invoke({'base_currency_value' : 10, 'conversion_rate' : 95.4559})

954.559

In [68]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
)

In [69]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [70]:
messages = [HumanMessage('What is the conversion factor between USD and INR, and based on that can you convert 10 USD to INR')]

In [71]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 USD to INR', additional_kwargs={}, response_metadata={})]

In [72]:
ai_message = llm_with_tools.invoke(messages)
messages.append(ai_message)

In [73]:
ai_message.tool_calls
#this gives conversion rate imbalanced not updated , as it didnt run the way expected 
#To handle this , we use Injected Tool Argument in langchain 


[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'e55a914t5',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': 'wj5p50s1m',
  'type': 'tool_call'}]

In [74]:
import json

for tool_call in ai_message.tool_calls:
    #execute the first tool and get the conversion rate 
    if tool_call['name'] == 'get_conversion_factor':
        tool_message1 = get_conversion_factor.invoke(tool_call)
        #fetch the conversion rate
        conversion_rate = json.loads(tool_message1.content)['conversion_rate']
        #append into the messages
        messages.append(tool_message1)
    #execute the 2nd tool using the conversion rate from tool 1
    if tool_call['name'] == 'convert':
        #fetch the current argument(adding key value pair in the existing one)
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_message2 = convert.invoke(tool_call)
        messages.append(tool_message2)

In [75]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 USD to INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'e55a914t5', 'function': {'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'name': 'get_conversion_factor'}, 'type': 'function'}, {'id': 'wj5p50s1m', 'function': {'arguments': '{"base_currency_value":10}', 'name': 'convert'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 347, 'total_tokens': 385, 'completion_time': 0.084318534, 'completion_tokens_details': None, 'prompt_time': 0.055146943, 'prompt_tokens_details': None, 'queue_time': 0.052891576, 'total_time': 0.139465477}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f3cc1-48f5-7f90-a97

In [77]:
llm_with_tools.invoke(messages).content

'The conversion factor between USD and INR is 95.4559. Based on this conversion factor, 10 USD is equivalent to 954.559 INR.'